In [60]:
import os
from functools import partial
from timeit import default_timer as timer
from datetime import timedelta
import ast
import math


import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape, Polygon
import pyproj
from shapely.ops import transform
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.plot import show
import tqdm
from shapely.geometry import box
from rasterio.errors import RasterioIOError
from rasterio.enums import Resampling


In [61]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf

In [62]:
target_years = list(range(2000, 2020))

In [63]:
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)

In [64]:

label_file_path = '/home/jan/Documents/EPFL/drivers/data/2024/cocoa_map_th_065.tif'

with rasterio.open(label_file_path) as src:
    label_src_profile = src.profile
    label_file_bounds = src.bounds
    #label_file_resolution = src.res <- wrong units, it's in degrees
label_file_extent = box(*label_file_bounds)

In [65]:
forestloss_tiles = forestloss_tiles.loc[forestloss_tiles.intersects(label_file_extent)]

In [66]:
'''Below I can confirm the resolution; replace the two numbers with width, height from label_src_profile

tmp_gdf = gpd.GeoDataFrame(geometry=[label_file_extent])
tmp_gdf.crs = 4326
tmp_gdf = tmp_gdf.to_crs(tmp_gdf.estimate_utm_crs())
( - tmp_gdf.bounds.minx + tmp_gdf.bounds.maxx)/108127 
( - tmp_gdf.bounds.miny + tmp_gdf.bounds.maxy)/75223
'''


'Below I can confirm the resolution; replace the two numbers with width, height from label_src_profile\n\ntmp_gdf = gpd.GeoDataFrame(geometry=[label_file_extent])\ntmp_gdf.crs = 4326\ntmp_gdf = tmp_gdf.to_crs(tmp_gdf.estimate_utm_crs())\n( - tmp_gdf.bounds.minx + tmp_gdf.bounds.maxx)/108127 \n( - tmp_gdf.bounds.miny + tmp_gdf.bounds.maxy)/75223\n'

In [67]:
label_file_resolution = (10,10)
target_patch_size = (1000,1000) #in meters, width, height


geoms = []
forest_loss_years = []


for idx, row in forestloss_tiles.iterrows():
    print(idx)

    path = os.path.join(forestloss_dir, row.filename)

    with rasterio.open(path) as src:
        forestloss_profile = src.profile
        
        
    #Define the window of label_file extent (to use for reading the TMF tile)
    label_file_window = rasterio.windows.from_bounds(left=label_file_bounds.left, 
                                          bottom=label_file_bounds.bottom, 
                                          right=label_file_bounds.right, 
                                          top=label_file_bounds.top,
                                          transform=forestloss_profile['transform'], 
                                         )
    
    
    #Read the TMF raster using the label_file window
    with rasterio.open(path) as src:
        forestloss_data = src.read(1, window=label_file_window)
        forestloss_profile = src.profile
        forestloss_bounds_orig = src.bounds


    forestloss_bounds = rasterio.coords.BoundingBox(
        left=max(forestloss_bounds_orig.left, label_file_bounds.left), 
        bottom=max(forestloss_bounds_orig.bottom, label_file_bounds.bottom), 
        right=min(forestloss_bounds_orig.right, label_file_bounds.right), 
        top=min(forestloss_bounds_orig.top, label_file_bounds.top)
    )

        
    with rasterio.open(label_file_path) as src:
        label_file_profile = src.profile #This is needed to compute the window

    #Read a part of the label_file raster corresponding to the intersection of it with the TMF tile
    window = rasterio.windows.from_bounds(left=forestloss_bounds.left, 
                                          bottom=forestloss_bounds.bottom, 
                                          right=forestloss_bounds.right, 
                                          top=forestloss_bounds.top,
                                          transform=label_file_profile['transform'], 
                                          #height=None, width=None, precision=None
                                         )

        
    with rasterio.open(label_file_path) as src:
        label_data = src.read( 
            out_shape=(
                src.count,
                forestloss_data.shape[0],
                forestloss_data.shape[1]
                ),
            resampling=Resampling.bilinear,
            window=window)
        label_data = label_data[0]
    

    
    if label_data.max() == 0: #Means there are no MapBiomas data for the area
        print(f"No label data for idx {idx}")
        continue

    rows, cols = forestloss_data.shape
    
    
    patch_rows, patch_cols = [int(math.ceil(size/res)) for size,res in zip(target_patch_size, label_file_resolution)]
    
    print("forestloss data shape: ", forestloss_data.shape)
    print("patch rows, cols: ", patch_rows, patch_cols)
    print("label_data  shape: ", label_data.shape)
    assert False
    

    for i in range(0, rows, patch_rows):
        for j in range(0, cols, patch_cols):
            forestloss_patch = forestloss_data[i:i+patch_rows, j:j+patch_cols]

            label_patch = label_data[i:i+patch_rows, j:j+patch_cols]
            
            if not np.isin(forestloss_patch, target_years).any(): #No forest loss in target period within the window
                continue
            if label_patch.max() == 0: #Means there are no positive examples for the window
                continue
            
            forest_loss_mask = (np.isin(forestloss_patch, target_years))

            lower_left_coords = rasterio.transform.xy(
                forestloss_profile['transform'], 
                i + round(max(0, label_file_window.row_off)), 
                j + round(max(0, label_file_window.col_off)),  #TODO: test this works as intended
                center='ll')
            upper_right_coords = rasterio.transform.xy(
                forestloss_profile['transform'], 
                i + round(max(0, label_file_window.row_off)) + patch_rows,
                j + round(max(0, label_file_window.col_off)) + patch_cols,  #TODO: test this works as intended
                center='ur')
            
            
            
            _geom = box(lower_left_coords[0], lower_left_coords[1], upper_right_coords[0], upper_right_coords[1])

            _forest_loss_years = np.unique(forestloss_patch[forest_loss_mask], return_counts=True)
            
            geoms.append(_geom)
            forest_loss_years.append(_forest_loss_years)

            
    print(f"Items: {len(geoms)}")


28
forestloss data shape:  (21061, 31910)
patch rows, cols:  100 100
label_data  shape:  (21061, 31910)


AssertionError: 

In [43]:
gdf = gpd.GeoDataFrame(geometry=geoms)

In [45]:
df = pd.DataFrame(gdf)
forest_loss_years_list = [ (item_tuple[0].tolist(),item_tuple[1].tolist()) for item_tuple in forest_loss_years]

df['forest_loss_years'] = forest_loss_years_list


In [47]:
df

,geometry,forest_loss_years
0,"POLYGON ((-8.57222 10.03746, -8.57222 10.01051...","([2001, 2002, 2004, 2006, 2010, 2011, 2013, 20..."
1,"POLYGON ((-8.54527 10.03746, -8.54527 10.01051...","([2002, 2005, 2007, 2011, 2013, 2015, 2016], [..."
2,"POLYGON ((-8.51832 10.03746, -8.51832 10.01051...","([2001, 2002, 2003, 2004, 2005, 2007, 2010, 20..."
3,"POLYGON ((-8.49137 10.03746, -8.49137 10.01051...","([2002, 2003, 2004, 2005, 2010, 2013, 2015, 20..."
4,"POLYGON ((-8.46442 10.03746, -8.46442 10.01051...","([2001, 2002, 2004, 2005, 2006, 2007, 2010, 20..."
...,...,...
50838,"POLYGON ((0.86252 5.77945, 0.86252 5.75250, 0....","([2010, 2011, 2013, 2014, 2015, 2016, 2017], [..."
50839,"POLYGON ((0.10793 5.75250, 0.10793 5.72555, 0....","([2002, 2013, 2019], [1, 1, 1])"
50840,"POLYGON ((0.08098 5.72555, 0.08098 5.69860, 0....","([2000, 2001, 2010, 2012, 2013, 2014, 2016, 20..."
50841,"POLYGON ((0.05403 5.69860, 0.05403 5.67165, 0....","([2000, 2010], [1, 1])"


In [48]:
df.to_csv('../data/cocoa.csv', index=False)

In [32]:
#gdf['drivers'] = gdf['drivers'].astype(str)
#gdf['forest_loss_years'] = gdf['forest_loss_years'].astype(str)


NameError: name 'df' is not defined

In [94]:
#gdf[['geometry', 'main_driver']].to_file('../../data/tmp/cocoa_2000_2020.gpkg')

/tmp/ipykernel_99284/3068935073.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf[['geometry', 'main_driver']].to_file('../../data/tmp/annotation_vancutsem_2000_2020.shp')


### Filtering

- above I extracted all possible labeled examples
- now filter by
    - "how main" the main driver is (e.g., at least 50% of pixels)
    - how much forest loss there was (e.g., at least 20 pixels)

In [49]:
path = '../data/cocoa.csv'
df = pd.read_csv(path)
gdf = gpd.GeoDataFrame(df)

In [50]:
gdf

,geometry,forest_loss_years
0,POLYGON ((-8.57221851447474 10.037460574402091...,"([2001, 2002, 2004, 2006, 2010, 2011, 2013, 20..."
1,POLYGON ((-8.545269055951154 10.03746057440209...,"([2002, 2005, 2007, 2011, 2013, 2015, 2016], [..."
2,POLYGON ((-8.518319597427569 10.03746057440209...,"([2001, 2002, 2003, 2004, 2005, 2007, 2010, 20..."
3,POLYGON ((-8.491370138903983 10.03746057440209...,"([2002, 2003, 2004, 2005, 2010, 2013, 2015, 20..."
4,POLYGON ((-8.464420680380398 10.03746057440209...,"([2001, 2002, 2004, 2005, 2006, 2007, 2010, 20..."
...,...,...
50838,POLYGON ((0.8625174200473587 5.779446127675559...,"([2010, 2011, 2013, 2014, 2015, 2016, 2017], [..."
50839,POLYGON ((0.1079325813869605 5.752496669151974...,"([2002, 2013, 2019], [1, 1, 1])"
50840,POLYGON ((0.0809831228633749 5.725547210628388...,"([2000, 2001, 2010, 2012, 2013, 2014, 2016, 20..."
50841,POLYGON ((0.0540336643397892 5.698597752104803...,"([2000, 2010], [1, 1])"


In [51]:
def convert_string_to_lists(input_string):
    try:
        lists = ast.literal_eval(input_string)
        if isinstance(lists, tuple) and len(lists) == 2 and all(isinstance(lst, list) for lst in lists):
            return lists
        else:
            return None
    except (SyntaxError, ValueError):
        return None

In [52]:
import re
import ast
def modify_string_multiple_items(input_string):
    # Adjusting regex to match and replace patterns for arrays with multiple items
    modified_string = re.sub(r'array\(\[([0-9, ]+)\](, dtype=\w+\d*)?\)', r'[\1]', input_string)
    # Removing any trailing unmatched parentheses
    modified_string = re.sub(r'\)+$', '', modified_string)
    modified_string = modified_string.replace('(','')
    return ast.literal_eval(modified_string)
    #return [eval(item.strip()) for item in  modified_string.split(',')]

# Testing with the new input
output_str_multiple = modify_string_multiple_items('(array([2017, 2020, 2021], dtype=uint16), array([2, 3, 5]))')
output_str_multiple

([2017, 2020, 2021], [2, 3, 5])

In [53]:
gdf['forest_loss_years'] = gdf['forest_loss_years'].apply(convert_string_to_lists)


In [54]:
def filter_forest_loss(item, min_m2_loss=2500, resolution=label_file_resolution):
    
    min_px_loss = int(math.ceil(min_m2_loss/(label_file_resolution[0]*label_file_resolution[1])))
    
    classes, counts = item
    if isinstance(classes, list):
        classes, counts = np.array(classes), np.array(counts) 
    return counts.sum() >= min_px_loss



In [55]:
gdf_filtered =  gdf.loc[gdf.forest_loss_years.apply(filter_forest_loss)]


In [56]:
gdf_filtered

,geometry,forest_loss_years
2,POLYGON ((-8.518319597427569 10.03746057440209...,"([2001, 2002, 2003, 2004, 2005, 2007, 2010, 20..."
3,POLYGON ((-8.491370138903983 10.03746057440209...,"([2002, 2003, 2004, 2005, 2010, 2013, 2015, 20..."
4,POLYGON ((-8.464420680380398 10.03746057440209...,"([2001, 2002, 2004, 2005, 2006, 2007, 2010, 20..."
5,POLYGON ((-8.437471221856812 10.03746057440209...,"([2000, 2001, 2002, 2003, 2004, 2006, 2010, 20..."
6,POLYGON ((-8.410521763333227 10.03746057440209...,"([2001, 2002, 2003, 2004, 2005, 2010, 2012, 20..."
...,...,...
50834,"POLYGON ((0.754719585953016 5.779446127675559,...","([2000, 2001, 2003, 2004, 2005, 2006, 2007, 20..."
50835,POLYGON ((0.7816690444766017 5.779446127675559...,"([2000, 2001, 2002, 2003, 2004, 2005, 2006, 20..."
50836,POLYGON ((0.8086185030001873 5.779446127675559...,"([2000, 2001, 2002, 2003, 2004, 2005, 2006, 20..."
50837,"POLYGON ((0.835567961523773 5.779446127675559,...","([2000, 2001, 2002, 2003, 2005, 2006, 2010, 20..."


In [57]:
gdf_filtered['driver'] = 'cocoa'


/tmp/ipykernel_18744/135524583.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gdf_filtered['driver'] = 'cocoa'


In [58]:
pd.DataFrame(gdf_filtered).to_csv('../data/cocoa_filtered.csv', index=False)


### Extract a small subset of the data to get started



In [59]:
subset = gdf_filtered.sample(1000, random_state=1)
pd.DataFrame(subset).to_csv('../data/dataset_small/cocoa_1000.csv', index=False)
#subset[['geometry', 'main_driver_name']].to_file('../../data/starting_dataset/cocoa_1000.gpkg')